In [0]:
%sql
SELECT o.channel,
    SUM(CASE WHEN o.order_date <= '2024-06-30' THEN l.line_revenue ELSE 0 END) AS fy2024_revenue,
    SUM(CASE WHEN o.order_date > '2024-06-30' THEN l.line_revenue ELSE 0 END) AS fy2025_revenue,
    ROUND((SUM(CASE WHEN o.order_date > '2024-06-30' THEN l.line_revenue ELSE 0 END) - 
           SUM(CASE WHEN o.order_date <= '2024-06-30' THEN l.line_revenue ELSE 0 END)) / 
           NULLIF(SUM(CASE WHEN o.order_date <= '2024-06-30' THEN l.line_revenue ELSE 0 END), 0) * 100, 2) AS growth_pct
FROM Sizwe_sales_lines l
INNER JOIN Sizwe_sales_orders o ON l.order_id = o.order_id
GROUP BY o.channel;

Both channels experienced solid growth of approximately 8.8% between FY2024 and FY2025. Wholesale grew from R308.2m to R335.4m, while Subscription grew from R132.8m to R144.6m, maintaining Wholesale's roughly 70% share of total revenue.

In [0]:
%sql
SELECT p.product_line,
    SUM(l.line_revenue) AS revenue,
    SUM(l.cogs) AS cogs,
    SUM(l.line_revenue - l.cogs) AS gross_profit,
    ROUND(SUM(l.line_revenue - l.cogs) / NULLIF(SUM(l.line_revenue), 0) * 100, 2) AS gross_margin_pct
FROM Sizwe_sales_lines l
INNER JOIN Sizwe_products p ON l.sku = p.sku
GROUP BY p.product_line
ORDER BY gross_profit DESC

House Blend earns the most total gross profit (R107.1m), but Single Origin earns the highest gross margin percentage (44.3% vs. House Blend's 38.3%). Therefore, the line that earns the most profit is not the same line that earns the highest margin.

In [0]:
%sql
SELECT c.channel,
    COUNT(a.invoice_no) AS total_invoices,
    COUNT(a.settlement_date) AS settled_invoices,
    COUNT(a.invoice_no) - COUNT(a.settlement_date) AS open_invoices,
    ROUND(AVG(datediff(a.settlement_date, a.invoice_date)), 2) AS avg_days_to_settle,
    SUM(CASE WHEN a.settlement_date IS NULL THEN a.invoice_amount ELSE 0 END) AS outstanding_value
FROM sizwe_ar_ledger a
INNER JOIN Sizwe_customers c ON a.customer_id = c.customer_id
GROUP BY c.channel

Wholesale accounts for a significantly larger volume of open invoices and total outstanding value, taking noticeably longer to settle compared to Subscription. Wholesale should be investigated first as the primary source of the collections slowdown.

In [0]:
%sql
SELECT  c.trading_name, c.segment, c.province,
    COALESCE(r.rep_name, 'Unassigned') AS sales_rep,
    SUM(l.line_revenue) AS fy2025_revenue
FROM Sizwe_customers c
INNER JOIN Sizwe_sales_orders o ON c.customer_id = o.customer_id
INNER JOIN Sizwe_sales_lines l ON o.order_id = l.order_id
LEFT JOIN Sizwe_reps r ON c.rep_id = r.rep_id
WHERE o.order_date > '2024-06-30'
GROUP BY c.trading_name, c.segment, c.province, r.rep_name
ORDER BY fy2025_revenue DESC
LIMIT 10

In [0]:
%sql
SELECT c.trading_name, c.segment, c.province,
    SUM(l.line_revenue) AS total_revenue
FROM Sizwe_customers c
INNER JOIN Sizwe_sales_orders o ON c.customer_id = o.customer_id
INNER JOIN Sizwe_sales_lines l ON o.order_id = l.order_id
WHERE c.channel = 'Wholesale'
  AND NOT EXISTS (
      SELECT 1 FROM Sizwe_trade_promotions tp 
      WHERE tp.customer_id = c.customer_id
  )
GROUP BY c.trading_name, c.segment, c.province
ORDER BY total_revenue DESC
LIMIT 15

I would recommend immediately engaging the top three wholesale customers with tailored trade promotions or commercial support to protect these high-value relationships from competitor poaching.

In [0]:
%sql
WITH rep_fy2025_rev AS (
    SELECT o.rep_id,
        SUM(l.line_revenue) AS total_fy2025_rev
    FROM Sizwe_sales_orders o
   INNER JOIN Sizwe_sales_lines l ON o.order_id = l.order_id
    WHERE o.order_date > '2024-06-30'
    GROUP BY o.rep_id
    )

SELECT r.rep_name,
    r.territory,
    r.target_monthly,
    ROUND(COALESCE(rev.total_fy2025_rev, 0) / 12.0, 2) AS actual_monthly_avg,
    ROUND((COALESCE(rev.total_fy2025_rev, 0) / 12.0) * 100.0 / NULLIF(r.target_monthly, 0), 2) AS pct_of_target
FROM Sizwe_reps r
LEFT JOIN rep_fy2025_rev rev ON r.rep_id = rev.rep_id
ORDER BY pct_of_target DESC

In [0]:
%sql
WITH monthly_rev AS (
    SELECT DATE_TRUNC('month', o.order_date) AS order_month,
     SUM(l.line_revenue) AS revenue
    FROM Sizwe_sales_orders o
    JOIN Sizwe_sales_lines l ON o.order_id = l.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
),
lagged_rev AS (
    SELECT order_month, revenue,
    LAG(revenue, 1) OVER (ORDER BY order_month) AS prev_month_revenue,
    AVG(revenue) OVER (ORDER BY order_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_3m_avg
    FROM monthly_rev
)
SELECT 
    TO_CHAR(order_month, 'yyyy-MM') AS month,
    ROUND(revenue, 2) AS revenue,
    ROUND((revenue - prev_month_revenue) * 100.0 / NULLIF(prev_month_revenue, 0), 2) AS mom_pct_change,
    ROUND(rolling_3m_avg, 2) AS rolling_3m_avg
FROM lagged_rev
ORDER BY order_month



There is a noticeable seasonal pattern with revenue dipping during mid-year winter months (such as June/July). Sizwe's quietest month across the two-year period is February.

In [0]:
%sql
WITH line_yearly AS (
    SELECT p.product_line,
        SUM(CASE WHEN o.order_date <= '2024-06-30' THEN l.line_revenue ELSE 0 END) AS rev_2024,
        SUM(CASE WHEN o.order_date <= '2024-06-30' THEN (l.line_revenue - l.cogs) ELSE 0 END) AS gp_2024,
        SUM(CASE WHEN o.order_date > '2024-06-30' THEN l.line_revenue ELSE 0 END) AS rev_2025,
        SUM(CASE WHEN o.order_date > '2024-06-30' THEN (l.line_revenue - l.cogs) ELSE 0 END) AS gp_2025
    FROM Sizwe_sales_lines l
    INNER JOIN Sizwe_products p ON l.sku = p.sku
    INNER JOIN Sizwe_sales_orders o ON l.order_id = o.order_id
    GROUP BY p.product_line
),
totals AS (
    SELECT 
        SUM(rev_2024) AS total_rev_2024,
        SUM(rev_2025) AS total_rev_2025
    FROM line_yearly
)
SELECT ly.product_line,
    ROUND(ly.rev_2024, 2) AS rev_2024,
    ROUND(ly.gp_2024 * 100.0 / NULLIF(ly.rev_2024, 0), 2) AS margin_2024_pct,
    ROUND(ly.rev_2024 * 100.0 / t.total_rev_2024, 2) AS share_2024_pct,
    ROUND(ly.rev_2025, 2) AS rev_2025,
    ROUND(ly.gp_2025 * 100.0 / NULLIF(ly.rev_2025, 0), 2) AS margin_2025_pct,
    ROUND(ly.rev_2025 * 100.0 / t.total_rev_2025, 2) AS share_2025_pct
FROM line_yearly ly
CROSS JOIN totals t
ORDER BY ly.rev_2025 DESC

The overall gross margin decline from 34.7% to 33.0% was driven by a shift in revenue mix toward lower-margin product lines and slight compression within specific lines. Specifically, Bulk Commercial (which carries a much lower margin of ~19.9%) increased its revenue share, while higher-margin lines like Single Origin experienced slight revenue share contraction.